In [1]:
import pandas as pd
import numpy as np
import math
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn import metrics

In [2]:
df_train_raw = pd.read_csv("training_pruebas.csv", low_memory=False)
df_train_raw

,ESTU_TIPODOCUMENTO,ESTU_NACIONALIDAD,ESTU_GENERO,ESTU_FECHANACIMIENTO,ESTU_EXTERIOR,PERIODO,ESTU_CONSECUTIVO,ESTU_ESTUDIANTE,ESTU_PAIS_RESIDE,ESTU_DEPTO_RESIDE,...,ESTU_PRIVADO_LIBERTAD,ESTU_COD_MCPIO_PRESENTACION,ESTU_MCPIO_PRESENTACION,ESTU_DEPTO_PRESENTACION,ESTU_COD_DEPTO_PRESENTACION,PUNT_GLOBAL,ESTU_INSE_INDIVIDUAL,ESTU_NSE_INDIVIDUAL,ESTU_NSE_IES,ESTU_ESTADOINVESTIGACION
0,CC,COLOMBIA,M,12/12/2000,NO,20231,EK202310215107,ESTUDIANTE,COLOMBIA,ATLANTICO,...,N,8001,BARRANQUILLA,ATLANTICO,8,188,55.570156,3.0,2,PUBLICAR
1,CC,COLOMBIA,M,21/07/2001,NO,20231,EK202310116775,ESTUDIANTE,COLOMBIA,CALDAS,...,N,17444,MARQUETALIA,CALDAS,17,140,38.105883,1.0,2,PUBLICAR
2,CC,COLOMBIA,F,21/04/1997,NO,20231,EK202310113020,ESTUDIANTE,COLOMBIA,TOLIMA,...,N,73268,ESPINAL,TOLIMA,73,140,65.658631,4.0,2,PUBLICAR
3,CC,COLOMBIA,F,20/07/1995,NO,20231,EK202310184031,ESTUDIANTE,COLOMBIA,LA GUAJIRA,...,N,44001,RIOHACHA,LA GUAJIRA,44,111,49.260729,2.0,2,PUBLICAR
4,CC,COLOMBIA,F,02/06/1978,NO,20231,EK202310177032,ESTUDIANTE,COLOMBIA,VALLE,...,N,76001,CALI,VALLE,76,135,55.138843,2.0,3,PUBLICAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111425,CC,COLOMBIA,F,08/02/2001,NO,20231,EK202310196332,ESTUDIANTE,COLOMBIA,BOYACA,...,N,15238,DUITAMA,BOYACA,15,137,NaN,NaN,2,PUBLICAR
111426,CC,COLOMBIA,F,06/10/2000,NO,20231,EK202310214172,ESTUDIANTE,COLOMBIA,VALLE,...,N,76622,ROLDANILLO,VALLE,76,161,63.619372,4.0,3,PUBLICAR
111427,CC,COLOMBIA,M,10/01/1965,NO,20231,EK202310223689,ESTUDIANTE,COLOMBIA,CESAR,...,N,20001,VALLEDUPAR,CESAR,20,124,48.648902,2.0,2,PUBLICAR
111428,CC,COLOMBIA,F,14/11/2001,NO,20231,EK202310147254,ESTUDIANTE,COLOMBIA,ATLANTICO,...,N,8001,BARRANQUILLA,ATLANTICO,8,142,52.877602,2.0,4,PUBLICAR


In [3]:
Columnas_no_Relevantes = [
    "ESTU_TIPODOCUMENTO", "ESTU_NACIONALIDAD","PERIODO","ESTU_CONSECUTIVO","ESTU_ESTUDIANTE","ESTU_PAIS_RESIDE",
    "ESTU_DEPTO_RESIDE","ESTU_MCPIO_RESIDE","ESTU_ESTADOCIVIL","ESTU_TIPODOCUMENTOSB11","FAMI_EDUCACIONPADRE","FAMI_EDUCACIONMADRE",
    "FAMI_OCUPACIONPADRE","FAMI_OCUPACIONMADRE","FAMI_TRABAJOLABORPADRE","FAMI_TRABAJOLABORMADRE","FAMI_CUANTOSCOMPARTEBAÑO",
    "ESTU_PRESENTACIONCASA","ESTU_PRESENTACIONSABADO","INST_NOMBRE_INSTITUCION","ESTU_PRGM_ACADEMICO","GRUPOREFERENCIA",
    "ESTU_PRGM_MUNICIPIO","ESTU_PRGM_DEPARTAMENTO","ESTU_NIVEL_PRGM_ACADEMICO","ESTU_NUCLEO_PREGRADO","ESTU_NUCLEO_PREGRADO_1",
    "ESTU_INST_MUNICIPIO","ESTU_INST_DEPARTAMENTO","INST_CARACTER_ACADEMICO","INST_ORIGEN","ESTU_MCPIO_PRESENTACION",
    "ESTU_DEPTO_PRESENTACION","ESTU_ESTADOINVESTIGACION", "ESTU_COD_RESIDE_DEPTO", "ESTU_COD_RESIDE_MCPIO","INST_COD_INSTITUCION",
    "ESTU_SNIES_PRGMACADEMICO","ESTU_PRGM_CODMUNICIPIO","ESTU_INST_CODMUNICIPIO","ESTU_COD_MCPIO_PRESENTACION","ESTU_COD_DEPTO_PRESENTACION"
]

df_train = df_train_raw.drop(columns = Columnas_no_Relevantes)

In [4]:
columnas_continuas = ["ESTU_INSE_INDIVIDUAL","ESTU_NSE_INDIVIDUAL", "ESTU_NSE_IES"]
columnas_categoricas = [col for col in df_train.columns if (col not in columnas_continuas) and (col != "PUNT_GLOBAL")]

def RellenarNull(data):
    new_data = data.copy()
    mean = new_data[columnas_continuas].mean()
    mode = new_data[columnas_categoricas].mode()

    # Llenar nulos con la media para variables numéricas
    for col in new_data[columnas_continuas]:
        for i, val in enumerate(new_data[col]):
            if math.isnan(val):
                new_data.loc[i, col] = mean[col]

    # Llenar nulos con la moda para variables categóricas
    new_data[columnas_categoricas] = new_data[columnas_categoricas].apply(lambda x: x.fillna(x.value_counts().index[0]))

    return new_data

df_train = RellenarNull(df_train)

for i, val in enumerate(df_train["PUNT_GLOBAL"]):
    if math.isnan(val):
        df_train.loc[i, col] = df_train["PUNT_GLOBAL"].mean()

In [5]:
def transformData(data):

    # Obtener edad actual de cada persona
    data["ESTU_FECHANACIMIENTO"] = pd.to_datetime(data["ESTU_FECHANACIMIENTO"], format="mixed", dayfirst=True, errors='coerce')
    today = pd.to_datetime('today', format="%d/%m/%Y")
    data["ESTU_FECHANACIMIENTO"] = data["ESTU_FECHANACIMIENTO"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))
    age_mean = data["ESTU_FECHANACIMIENTO"].mean()

    for i, val in enumerate(data["ESTU_FECHANACIMIENTO"]):
        if str(val) == 'nan':
            data.loc[i, "ESTU_FECHANACIMIENTO"] = age_mean

    # Trasnformar valores binomiales

    # Genero (M: 0, F: 1)
    data["ESTU_GENERO"] = data["ESTU_GENERO"].apply(lambda x: 1 if x == 'F' else 0)

    # Estudia en el exterior (NO: False, SI: True)
    data["ESTU_EXTERIOR"] = data["ESTU_EXTERIOR"].apply(lambda x: x == 'SI')

    # Área de residencia (Area Rural: 0, Cabecera Municipal: 1)
    data["ESTU_AREARESIDE"] = data["ESTU_AREARESIDE"].apply(lambda x: 1 if x == 'Cabecera Municipal' else 0)

    # Pago con beca
    data["ESTU_PAGOMATRICULABECA"] = data["ESTU_PAGOMATRICULABECA"].apply(lambda x: x == 'Si')
    # Pago con crédito
    data["ESTU_PAGOMATRICULACREDITO"] = data["ESTU_PAGOMATRICULACREDITO"].apply(lambda x: x == 'Si')
    # Pago con padres
    data["ESTU_PAGOMATRICULAPADRES"] = data["ESTU_PAGOMATRICULAPADRES"].apply(lambda x: x == 'Si')
    # Pago propio
    data["ESTU_PAGOMATRICULAPROPIO"] = data["ESTU_PAGOMATRICULAPROPIO"].apply(lambda x: x == 'Si')

    # Hizo simulacro
    data["ESTU_SIMULACROTIPOICFES"] = data["ESTU_SIMULACROTIPOICFES"].apply(lambda x: x == 'Si')
    # Hizo actividades de refuerzo en áreas
    data["ESTU_ACTIVIDADREFUERZOAREAS"] = data["ESTU_ACTIVIDADREFUERZOAREAS"].apply(lambda x: x == 'Si')
    # Hizo actividades de refuerzo genéricas
    data["ESTU_ACTIVIDADREFUERZOGENERIC"] = data["ESTU_ACTIVIDADREFUERZOGENERIC"].apply(lambda x: x == 'Si')

    # Tiene internet
    data["FAMI_TIENEINTERNET"] = data["FAMI_TIENEINTERNET"].apply(lambda x: x == 'Si')
    # Tiene computador
    data["FAMI_TIENECOMPUTADOR"] = data["FAMI_TIENECOMPUTADOR"].apply(lambda x: x == 'Si')
    # Tiene lavadora
    data["FAMI_TIENELAVADORA"] = data["FAMI_TIENELAVADORA"].apply(lambda x: x == 'Si')
    # Tiene microondas
    data["FAMI_TIENEHORNOMICROOGAS"] = data["FAMI_TIENEHORNOMICROOGAS"].apply(lambda x: x == 'Si')
    # Tiene TV
    data["FAMI_TIENESERVICIOTV"] = data["FAMI_TIENESERVICIOTV"].apply(lambda x: x == 'Si')
    # Tiene automovil
    data["FAMI_TIENEAUTOMOVIL"] = data["FAMI_TIENEAUTOMOVIL"].apply(lambda x: x == 'Si')
    # Tiene motocicleta
    data["FAMI_TIENEMOTOCICLETA"] = data["FAMI_TIENEMOTOCICLETA"].apply(lambda x: x == 'Si')
    # Tiene consola de videojuegos
    data["FAMI_TIENECONSOLAVIDEOJUEGOS"] = data["FAMI_TIENECONSOLAVIDEOJUEGOS"].apply(lambda x: x == 'Si')

    # Método del programa (PRESENCIAL: 0 DISTANCIA VITUAL: 1)
    data["ESTU_METODO_PRGM"] = data["ESTU_METODO_PRGM"].apply(lambda x: 1 if x == 'DISTANCIA VITUAL' else 0)

    # Privado de libertad
    data["ESTU_PRIVADO_LIBERTAD"] = data["ESTU_PRIVADO_LIBERTAD"].apply(lambda x: x == 'S')

transformData(df_train)
print("Nulos: ", np.where(pd.isnull(df_train)))

Nulos:  (array([], dtype=int64), array([], dtype=int64))


In [6]:
def normalizar(data):
    for i, col in enumerate(data.columns):
        data[col] = 0 if data[col].std() == 0 else (data[col] - data[col].mean()) / data[col].std()

# Crear dummies
df_train_dummies = pd.get_dummies(df_train)
normalizar(df_train_dummies)
df_train_dummies

,ESTU_GENERO,ESTU_FECHANACIMIENTO,ESTU_EXTERIOR,ESTU_AREARESIDE,ESTU_PAGOMATRICULABECA,ESTU_PAGOMATRICULACREDITO,ESTU_PAGOMATRICULAPADRES,ESTU_PAGOMATRICULAPROPIO,ESTU_SIMULACROTIPOICFES,ESTU_ACTIVIDADREFUERZOAREAS,...,ESTU_PAGOMATRICULA_Becas o Subsidios,ESTU_PAGOMATRICULA_Creditos,ESTU_PAGOMATRICULA_Familiares,ESTU_VLRULTIMOSEMESCURSADO_Entre 3 y 5 millones,ESTU_VLRULTIMOSEMESCURSADO_Entre 5 y 7 millones,ESTU_VLRULTIMOSEMESCURSADO_Entre 500 mil y un millon de pesos,ESTU_VLRULTIMOSEMESCURSADO_Entre un millon y 3 millones de pesose,ESTU_VLRULTIMOSEMESCURSADO_Mas de 7 millones,ESTU_VLRULTIMOSEMESCURSADO_Menos de 500 mil pesos,ESTU_VLRULTIMOSEMESCURSADO_No pago semestre
0,-1.141938,-0.709054,-0.123835,0.37177,1.534565,-0.617858,-0.834938,-0.942998,0.075645,0.098651,...,-0.507153,-0.547684,1.485369,-0.456533,-0.273497,-0.239647,-0.704445,-0.276743,2.582217,-0.453214
1,-1.141938,-0.852607,-0.123835,0.37177,1.534565,-0.617858,-0.834938,-0.942998,0.075645,0.098651,...,1.971773,-0.547684,-0.673227,-0.456533,-0.273497,-0.239647,-0.704445,-0.276743,2.582217,-0.453214
2,0.875696,-0.134843,-0.123835,0.37177,1.534565,-0.617858,-0.834938,-0.942998,0.075645,0.098651,...,1.971773,-0.547684,-0.673227,-0.456533,-0.273497,-0.239647,-0.704445,-0.276743,2.582217,-0.453214
3,0.875696,0.008710,-0.123835,0.37177,1.534565,-0.617858,1.197683,-0.942998,0.075645,0.098651,...,1.971773,-0.547684,-0.673227,-0.456533,-0.273497,-0.239647,-0.704445,-0.276743,-0.387261,2.206445
4,0.875696,2.449108,-0.123835,0.37177,-0.651645,-0.617858,-0.834938,1.060439,0.075645,0.098651,...,-0.507153,-0.547684,-0.673227,2.190403,-0.273497,-0.239647,-0.704445,-0.276743,-0.387261,-0.453214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111425,0.875696,-0.709054,-0.123835,0.37177,1.534565,-0.617858,1.197683,-0.942998,0.075645,0.098651,...,-0.507153,-0.547684,1.485369,-0.456533,-0.273497,-0.239647,1.419545,-0.276743,-0.387261,-0.453214
111426,0.875696,-0.709054,-0.123835,0.37177,1.534565,-0.617858,1.197683,-0.942998,0.075645,0.098651,...,-0.507153,-0.547684,1.485369,2.190403,-0.273497,-0.239647,-0.704445,-0.276743,-0.387261,-0.453214
111427,-1.141938,4.458848,-0.123835,0.37177,-0.651645,-0.617858,-0.834938,1.060439,0.075645,0.098651,...,-0.507153,1.825854,-0.673227,-0.456533,-0.273497,-0.239647,1.419545,-0.276743,-0.387261,-0.453214
111428,0.875696,-0.852607,-0.123835,0.37177,-0.651645,1.618480,-0.834938,-0.942998,0.075645,0.098651,...,-0.507153,1.825854,-0.673227,2.190403,-0.273497,-0.239647,-0.704445,-0.276743,-0.387261,-0.453214


In [7]:
df_train_dummies.corrwith(df_train_dummies["PUNT_GLOBAL"]).sort_values(ascending=False).iloc[1:]

ESTU_NSE_IES                                                             0.262818
ESTU_VALORMATRICULAUNIVERSIDAD_Más de 7 millones                         0.244534
ESTU_INSE_INDIVIDUAL                                                     0.238292
ESTU_NSE_INDIVIDUAL                                                      0.215721
ESTU_VLRULTIMOSEMESCURSADO_Mas de 7 millones                             0.213380
                                                                           ...   
ESTU_VALORMATRICULAUNIVERSIDAD_Entre 1 millón y menos de 2.5 millones   -0.144575
FAMI_ESTRATOVIVIENDA_Estrato 1                                          -0.160728
ESTU_PAGOMATRICULAPROPIO                                                -0.165790
ESTU_VLRULTIMOSEMESCURSADO_Entre un millon y 3 millones de pesose       -0.185439
ESTU_FECHANACIMIENTO                                                    -0.252638
Length: 82, dtype: float64

**Divida los datos en training y testing**

In [8]:
x, y = df_train_dummies[[col for col in df_train_dummies.columns if col != "PUNT_GLOBAL"]], df_train["PUNT_GLOBAL"]
xr, xs, yr, ys = train_test_split(x, y, test_size=0.2, random_state=42)

In [9]:
x, y = df_train_dummies[[col for col in df_train_dummies.columns if col != "PUNT_GLOBAL"]], (df_train["PUNT_GLOBAL"] > 172).astype(int)

In [10]:
est_test_raw = pd.read_csv("test_pruebas.csv", low_memory=False)
est_test = est_test_raw.drop(columns=Columnas_no_Relevantes)
est_test = RellenarNull(est_test)
transformData(est_test)
est_test_dumm = pd.get_dummies(est_test)
normalizar(est_test_dumm)
est_test_dumm
def get_r2(true, pred):
    SSR, SST = sum((true-pred)**2), sum((true-np.mean(true))**2)
    R2 = 1 - (float(SSR))/SST
    return R2

def get_mse(true, pred):
    return (1/len(true))*sum((true-pred)**2)

def get_mape(true, pred):
    return (100/len(true)*sum(abs(pred-true)/true))

In [11]:
xr, xs, yr, ys = train_test_split(x, y, test_size=0.2, random_state=42)

lgr = LogisticRegression()
lgr.fit(xr, yr)

yr_hat = lgr.predict(xr)

print("Regresión logítisca")
print("MSE(train):", get_mse(yr_hat, yr))

ys_hat = lgr.predict(xs)

print("MSE(test):", get_mse(ys_hat, ys))

Regresión logítisca
MSE(train): 0.11046172484968142
MSE(test): 0.11267163241496904


In [12]:
importance = pd.DataFrame({'Variable': est_test_dumm.columns, 'Coeficiente': lgr.coef_[0]}).sort_values(ascending=False, by=["Coeficiente"]).iloc[1:]
importance

,Variable,Coeficiente
21,ESTU_INSE_INDIVIDUAL,0.266110
81,ESTU_VLRULTIMOSEMESCURSADO_No pago semestre,0.195276
3,ESTU_AREARESIDE,0.194059
14,FAMI_TIENEHORNOMICROOGAS,0.165948
12,FAMI_TIENECOMPUTADOR,0.164255
...,...,...
15,FAMI_TIENESERVICIOTV,-0.155701
0,ESTU_GENERO,-0.207305
17,FAMI_TIENEMOTOCICLETA,-0.213083
59,FAMI_ESTRATOVIVIENDA_Estrato 1,-0.223989


In [13]:
conf_r = pd.DataFrame(metrics.confusion_matrix(yr, yr_hat))
print("Filas: Real, Columnas: Predicción\n")
print("Matríz de confusión (Train)")
conf_r

Filas: Real, Columnas: Predicción

Matríz de confusión (Train)


,0,1
0,77562,1164
1,8683,1735


In [14]:
pre_r, acc_r, rec_r = precision_score(yr, yr_hat), accuracy_score(yr, yr_hat), recall_score(yr, yr_hat)
print("PRECISION:",pre_r)
print("ACCURACY:",acc_r)
print("RECALL:", rec_r)

PRECISION: 0.5984822352535357
ACCURACY: 0.8895382751503186
RECALL: 0.16653868304856978


In [15]:
conf_s = pd.DataFrame(metrics.confusion_matrix(ys, ys_hat))
print("Matríz de confusión (Test)")
conf_s

Matríz de confusión (Test)


,0,1
0,19321,305
1,2206,454


In [16]:
pre_s, acc_s, rec_s = precision_score(ys, ys_hat), accuracy_score(ys, ys_hat), recall_score(ys, ys_hat)
print("PRECISION:",pre_s)
print("ACCURACY:",acc_s)
print("RECALL:", rec_s)

PRECISION: 0.5981554677206851
ACCURACY: 0.887328367585031
RECALL: 0.17067669172932332
